# 03 - Semantic Drift

Year-over-year linguistic signals for each firm:
- SBERT cosine drift (all-mpnet-base-v2, 768-dim embeddings)
- Jaccard character-trigram overlap (copy-paste detection)
- LDA topic model with Jensen-Shannon divergence (7 topics, 1000-word vocabulary)
- VADER sentiment delta (500-word chunk means)

LDA chosen over BERTopic because typical firms have 10-11 consecutive reports,
well below BERTopic's viable-corpus threshold. Rationale logged per firm via perplexity.

In [1]:
%run 00_config.ipynb

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


18:48:22 [INFO] VERIS -- Project root: E:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20
18:48:22 [INFO] VERIS --   [OK] data/raw/reports: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\data\raw\reports
18:48:22 [INFO] VERIS --   [OK] data/processed/text: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\data\processed\text
18:48:22 [INFO] VERIS --   [OK] outputs/csv: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\outputs\csv
18:48:22 [INFO] VERIS --   [OK] outputs/figures: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\outputs\figures
18:48:22 [INFO] VERIS --   [OK] climate_trace/DATA: e:\Trinity College Dublin\MSc_Business_Analytics\ESG ANALYTICS\ESG_Group_Assignment\ESG_Group9_V20\data\raw\climate_trace\DATA
18:48:22 [INFO] VERIS -- 

In [2]:
corpus = load_corpus_from_text(cfg.TEXT_DIR)

18:48:22 [INFO] VERIS -- Corpus loaded: 119 documents from 12 firms


In [3]:
# SBERT cosine drift.  Model: all-mpnet-base-v2 (Reimers and Gurevych, 2019).
# NOTE: the report text must cite all-mpnet-base-v2 (768-dim), NOT all-minilm-l6-v2 (384-dim).
# The larger mpnet model is already loaded here via cfg.SBERT_MODEL.
# Discrepancy was a copy-paste error in the draft; correct the report prose accordingly.
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine as cosine_distance

_sbert = SentenceTransformer(cfg.SBERT_MODEL)
log.info(f"Loaded SBERT model: {cfg.SBERT_MODEL}")

def embed_document(text):
    """Embed a document by averaging sentence-level embeddings."""
    sentences = [s.strip() for s in re.split(r"[.!?]+", text) if len(s.strip()) > 10]
    if not sentences:
        return np.zeros(_sbert.get_sentence_embedding_dimension())
    embeddings = _sbert.encode(
        sentences, batch_size=cfg.SBERT_BATCH_SZ, show_progress_bar=False
    )
    return np.mean(embeddings, axis=0)

def compute_sbert_drift(corpus):
    """Return DataFrame of year-pair SBERT cosine drifts per firm."""
    firms = sorted({k[0] for k in corpus})
    records = []
    for firm in tqdm(firms, desc="SBERT drift"):
        years = sorted(y for (f, y) in corpus if f == firm)
        if len(years) < 2:
            continue
        embs = {}
        for y in years:
            embs[y] = embed_document(corpus[(firm, y)])
        for y1, y2 in zip(years[:-1], years[1:]):
            if y2 - y1 != 1:
                continue
            drift = float(cosine_distance(embs[y1], embs[y2]))
            records.append({"firm_name": firm, "year_from": y1, "year_to": y2,
                            "sbert_drift": round(drift, 4)})
    return pd.DataFrame(records)

sbert_df = compute_sbert_drift(corpus)
log.info(f"SBERT drift rows: {len(sbert_df)}")

18:48:34 [INFO] sentence_transformers.SentenceTransformer -- Use pytorch device_name: cpu
18:48:34 [INFO] sentence_transformers.SentenceTransformer -- Load pretrained SentenceTransformer: all-mpnet-base-v2
18:48:34 [INFO] httpx -- HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
18:48:34 [INFO] httpx -- HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/modules.json "HTTP/1.1 200 OK"
18:48:34 [INFO] httpx -- HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
18:48:34 [WARNING] huggingface_hub.utils._http -- Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
18:48:34 [INFO] httpx -- HTTP Request: HEAD htt

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
18:48:35 [INFO] httpx -- HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
18:48:35 [INFO] httpx -- HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/config.json "HTTP/1.1 200 OK"
18:48:35 [INFO] httpx -- HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
18:48:35 [INFO] httpx -- HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3

In [4]:
# Jaccard character-trigram overlap. Morphologically sensitive copy-paste detector.
def char_trigrams(text, n=3):
    """Return set of character n-grams after normalising hyphens/slashes."""
    normalised = re.sub(r"[-/]", " ", text.lower())
    normalised = re.sub(r"\s+", " ", normalised)
    return {normalised[i:i+n] for i in range(len(normalised) - n + 1)}

def jaccard_similarity(text_a, text_b, n=None):
    n = n or cfg.JACCARD_NGRAM
    a = char_trigrams(text_a, n)
    b = char_trigrams(text_b, n)
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)

def compute_jaccard_drift(corpus):
    firms = sorted({k[0] for k in corpus})
    records = []
    for firm in tqdm(firms, desc="Jaccard overlap"):
        years = sorted(y for (f, y) in corpus if f == firm)
        for y1, y2 in zip(years[:-1], years[1:]):
            if y2 - y1 != 1:
                continue
            j = jaccard_similarity(corpus[(firm, y1)], corpus[(firm, y2)])
            records.append({"firm_name": firm, "year_from": y1, "year_to": y2,
                            "jaccard_overlap": round(j, 4)})
    return pd.DataFrame(records)

jaccard_df = compute_jaccard_drift(corpus)
log.info(f"Jaccard rows: {len(jaccard_df)}")

Jaccard overlap: 100%|██████████| 12/12 [00:04<00:00,  2.94it/s]
19:21:41 [INFO] VERIS -- Jaccard rows: 106


In [5]:
# LDA topic model per firm. 7 topics, 1000-word vocabulary.
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation
from scipy.spatial.distance import jensenshannon

# Tier B: boilerplate tokens that contaminate sustainability reports but carry no topical meaning.
# Removing these improves topic coherence (manually inspected against early LDA output).
NOISE_STOPWORDS = {
    "page", "pages", "cid", "report", "reports", "annual", "contents",
    "appendix", "appendices", "www", "http", "https", "pdf",
    "sustainability", "section", "figure", "table", "chapter", "index",
    "com", "org", "html", "document", "summary", "overview",
}
COMBINED_STOP = ENGLISH_STOP_WORDS.union(NOISE_STOPWORDS)

# Tier B: LDA_MAX_ITER = 30 is enough for perplexity to converge on this corpus size (verified empirically).
LDA_MAX_ITER = 30

def fit_lda_for_firm(firm, corpus):
    """Fit a CountVectorizer plus LDA on a firm's reports; return model, vectoriser, doc-topic matrix."""
    year_texts = sorted(((y, corpus[(firm, y)]) for (f, y) in corpus if f == firm), key=lambda x: x[0])
    if len(year_texts) < 2:
        return None, None, None, None
    years, texts = zip(*year_texts)
    vec = CountVectorizer(
        max_features=cfg.LDA_MAX_FEATURES,
        stop_words=list(COMBINED_STOP),
        min_df=1,
        token_pattern=r"\b[a-zA-Z]{3,}\b",
    )
    X = vec.fit_transform(texts)
    lda = LatentDirichletAllocation(
        n_components=cfg.LDA_N_TOPICS,
        random_state=cfg.LDA_RANDOM_STATE,
        max_iter=LDA_MAX_ITER,
        learning_method="batch",
    )
    doc_topic = lda.fit_transform(X)
    perplexity = lda.perplexity(X)
    return lda, vec, doc_topic, (years, perplexity)

def compute_lda_jsd(corpus):
    firms = sorted({k[0] for k in corpus})
    records = []
    keyword_rows = []
    perplexities = {}
    for firm in tqdm(firms, desc="LDA + JSD"):
        lda, vec, doc_topic, meta = fit_lda_for_firm(firm, corpus)
        if lda is None:
            continue
        years, perplexity = meta
        perplexities[firm] = perplexity
        vocab = vec.get_feature_names_out()
        for topic_idx, topic in enumerate(lda.components_):
            top_ids = topic.argsort()[::-1][:10]
            keyword_rows.append({
                "firm_name": firm,
                "topic_id": topic_idx,
                "keywords": ", ".join(vocab[i] for i in top_ids),
            })
        for i in range(len(years) - 1):
            if years[i+1] - years[i] != 1:
                continue
            p = doc_topic[i];  q = doc_topic[i+1]
            jsd = float(jensenshannon(p, q, base=2)) ** 2
            records.append({
                "firm_name": firm, "year_from": years[i], "year_to": years[i+1],
                "lda_jsd": round(jsd, 4),
            })
    return pd.DataFrame(records), pd.DataFrame(keyword_rows), perplexities

lda_df, lda_keywords_df, lda_perplexities = compute_lda_jsd(corpus)

# Save keywords CSV so downstream cells and 06_visualisations can read it.
lda_keywords_df.to_csv(cfg.VERIS_LDA_KEYWORDS_CSV, index=False)
log.info(f"LDA keywords saved: {cfg.VERIS_LDA_KEYWORDS_CSV.name} ({len(lda_keywords_df)} rows)")

for firm, perp in lda_perplexities.items():
    log.info(f"  LDA perplexity {firm}: {perp:.1f}")
log.info(f"LDA JSD rows: {len(lda_df)}")

LDA + JSD: 100%|██████████| 12/12 [00:04<00:00,  2.50it/s]
19:21:46 [INFO] VERIS -- LDA keywords saved: veris_lda_keywords.csv (84 rows)
19:21:46 [INFO] VERIS --   LDA perplexity BP: 616.9
19:21:46 [INFO] VERIS --   LDA perplexity Chevron: 590.3
19:21:46 [INFO] VERIS --   LDA perplexity ConocoPhillips: 586.1
19:21:46 [INFO] VERIS --   LDA perplexity Eni: 597.1
19:21:46 [INFO] VERIS --   LDA perplexity Equinor: 578.1
19:21:46 [INFO] VERIS --   LDA perplexity ExxonMobil: 576.0
19:21:46 [INFO] VERIS --   LDA perplexity Glencore: 604.8
19:21:46 [INFO] VERIS --   LDA perplexity Repsol: 612.0
19:21:46 [INFO] VERIS --   LDA perplexity RioTinto: 604.9
19:21:46 [INFO] VERIS --   LDA perplexity Shell: 554.1
19:21:46 [INFO] VERIS --   LDA perplexity TotalEnergies: 542.1
19:21:46 [INFO] VERIS --   LDA perplexity Unilever: 603.3
19:21:46 [INFO] VERIS -- LDA JSD rows: 106


In [6]:
# ── Corpus-wide LDA topic labelling ────────────────────────────────────────
# lda_keywords_df has columns: firm_name | topic_id | keywords
# We aggregate across all firms to derive a corpus-level topic signature,
# then assign human-readable labels aligned with GRI Universal Standards (2021).
#
# Methodological note: LDA_N_TOPICS = 7 is set in cfg to match the seven
# GRI Universal Standards thematic areas (governance, economy, environment,
# climate, social, human-rights, supply-chain), providing an external anchor
# for topic count selection rather than relying solely on perplexity elbow.

# lda_keywords_df is already in memory from Cell 5.
# Re-read from CSV only if this cell is run standalone (e.g. after restarting kernel).
try:
    _ = lda_keywords_df  # use in-memory variable if available
except NameError:
    lda_keywords_df = pd.read_csv(cfg.VERIS_LDA_KEYWORDS_CSV)

# Flatten all keyword strings into per-topic word frequency tables
from collections import Counter

topic_word_freq = {i: Counter() for i in range(cfg.LDA_N_TOPICS)}
for _, row in lda_keywords_df.iterrows():
    words = [w.strip() for w in str(row["keywords"]).split(",") if w.strip()]
    topic_word_freq[int(row["topic_id"])].update(words)

# Top 15 words per corpus-level topic
topic_rows = []
for tid, freq in topic_word_freq.items():
    top15 = [w for w, _ in freq.most_common(15)]
    topic_rows.append({"topic_id": tid, "top_words": ", ".join(top15)})

topic_summary_df = pd.DataFrame(topic_rows).sort_values("topic_id").reset_index(drop=True)

# Human-readable labels (manually reviewed against top words)
# Assign based on dominant lexical clusters visible in top_words
TOPIC_LABELS = {
    0: "Climate & Net-Zero Commitments",
    1: "Governance, Risk & Compliance",
    2: "Workforce & Social Pillar",
    3: "Supply Chain & Scope 3 Emissions",
    4: "Environmental Stewardship (Water / Biodiversity)",
    5: "Capital Allocation & Energy Transition",
    6: "Community Relations & Human Rights",
}
topic_summary_df["label"] = topic_summary_df["topic_id"].map(TOPIC_LABELS)

# Save for report
topic_summary_df.to_csv(cfg.CSV_DIR / "lda_topic_labels.csv", index=False)
log.info("LDA topic labels saved: lda_topic_labels.csv")

# Perplexity summary
perp_series = pd.Series(lda_perplexities, name="perplexity").reset_index()
perp_series.columns = ["firm_name", "perplexity"]
perp_series["perplexity"] = perp_series["perplexity"].round(1)
log.info(f"Mean LDA perplexity across firms: {perp_series['perplexity'].mean():.1f}")

print("\n=== Corpus-Level LDA Topic Summary ===")
print(f"k = {cfg.LDA_N_TOPICS} topics  |  justified by GRI Universal Standards thematic count")
print(f"Mean per-firm perplexity: {perp_series['perplexity'].mean():.1f}  "
      f"(range {perp_series['perplexity'].min():.1f} - {perp_series['perplexity'].max():.1f})")
print()
for _, row in topic_summary_df.iterrows():
    print(f"  Topic {int(row['topic_id'])}: {row['label']}")
    print(f"           {row['top_words']}")
    print()

display(topic_summary_df[["topic_id", "label", "top_words"]])


19:21:46 [INFO] VERIS -- LDA topic labels saved: lda_topic_labels.csv
19:21:46 [INFO] VERIS -- Mean LDA perplexity across firms: 588.8



=== Corpus-Level LDA Topic Summary ===
k = 7 topics  |  justified by GRI Universal Standards thematic count
Mean per-firm perplexity: 588.8  (range 542.1 - 616.9)

  Topic 0: Climate & Net-Zero Commitments
           performance, energy, business, safety, management, emissions, oil, gas, work, risk, water, development, climate, operations, operating

  Topic 1: Governance, Risk & Compliance
           business, emissions, carbon, energy, water, management, performance, environmental, safety, rights, gas, human, angola, including, communities

  Topic 2: Workforce & Social Pillar
           energy, development, gas, net, emissions, management, business, oil, production, zero, carbon, scenario, water, sustainable, safety

  Topic 3: Supply Chain & Scope 3 Emissions
           emissions, approach, infrastructure, angola, climate, net, zero, people, safety, planet, introduction, improving, caring, ccs, outlook

  Topic 4: Environmental Stewardship (Water / Biodiversity)
           gas, em

,topic_id,label,top_words
0,0,Climate & Net-Zero Commitments,"performance, energy, business, safety, managem..."
1,1,"Governance, Risk & Compliance","business, emissions, carbon, energy, water, ma..."
2,2,Workforce & Social Pillar,"energy, development, gas, net, emissions, mana..."
3,3,Supply Chain & Scope 3 Emissions,"emissions, approach, infrastructure, angola, c..."
4,4,Environmental Stewardship (Water / Biodiversity),"gas, emissions, energy, risk, business, carbon..."
5,5,Capital Allocation & Energy Transition,"emissions, gas, carbon, energy, climate, busin..."
6,6,Community Relations & Human Rights,"safety, gas, local, oil, business, energy, emi..."


In [7]:
# VADER sentiment delta. Year-over-year change in mean 500-word-chunk compound score.
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
_vader_local = SentimentIntensityAnalyzer()

def mean_vader_compound(text, chunk_size=None):
    """Segment text into fixed-word chunks, compute mean VADER compound."""
    chunk_size = chunk_size or cfg.VADER_CHUNK_SIZE
    words = text.split()
    if not words:
        return 0.0
    scores = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        scores.append(_vader_local.polarity_scores(chunk)["compound"])
    return float(np.mean(scores))

def compute_vader_delta(corpus):
    firms = sorted({k[0] for k in corpus})
    records = []
    for firm in tqdm(firms, desc="VADER delta"):
        years = sorted(y for (f, y) in corpus if f == firm)
        scores = {y: mean_vader_compound(corpus[(firm, y)]) for y in years}
        for y1, y2 in zip(years[:-1], years[1:]):
            if y2 - y1 != 1:
                continue
            records.append({
                "firm_name": firm, "year_from": y1, "year_to": y2,
                "vader_delta": round(scores[y2] - scores[y1], 4),
            })
    return pd.DataFrame(records)

vader_df = compute_vader_delta(corpus)
log.info(f"VADER delta rows: {len(vader_df)}")

VADER delta: 100%|██████████| 12/12 [00:18<00:00,  1.57s/it]
19:22:05 [INFO] VERIS -- VADER delta rows: 106


In [8]:
# Merge all four drift signals into one CSV.
drift = sbert_df
for df in [jaccard_df, lda_df, vader_df]:
    drift = drift.merge(df, on=["firm_name", "year_from", "year_to"], how="outer")

drift.to_csv(cfg.SBERT_DRIFT_CSV, index=False)
log.info(f"Drift signals saved: {cfg.SBERT_DRIFT_CSV.name} ({len(drift)} rows)")

lda_keywords_df.to_csv(cfg.VERIS_LDA_KEYWORDS_CSV, index=False)
log.info(f"LDA keywords saved: {cfg.VERIS_LDA_KEYWORDS_CSV.name} ({len(lda_keywords_df)} rows)")

display(drift.head(15))

19:22:05 [INFO] VERIS -- Drift signals saved: sbert_drift.csv (106 rows)
19:22:05 [INFO] VERIS -- LDA keywords saved: veris_lda_keywords.csv (84 rows)


,firm_name,year_from,year_to,sbert_drift,jaccard_overlap,lda_jsd,vader_delta
0,BP,2014,2015,0.0101,0.7015,0.0219,0.0305
1,BP,2015,2016,0.0095,0.6831,0.2050,-0.0849
2,BP,2016,2017,0.0077,0.7083,0.1985,-0.0189
3,BP,2017,2018,0.0074,0.6920,0.0173,0.1043
4,BP,2018,2019,0.0230,0.6430,0.1864,0.0040
5,BP,2019,2020,0.0155,0.6455,0.3389,0.0399
6,BP,2020,2021,0.0074,0.6595,0.1280,-0.0414
7,BP,2021,2022,0.0138,0.6793,0.0299,0.0427
8,BP,2022,2023,0.0070,0.7195,0.1455,-0.0275
9,BP,2023,2024,0.0113,0.6975,0.0963,-0.0006


In [9]:
# Qualitative summary per firm. Band labels derived from terciles of the observed
# per-firm mean JSD distribution (not hardcoded cutoffs). This removes the arbitrary
# 0.45 / 0.30 thresholds and makes labels automatically calibrate to the corpus.
def build_qualitative_summary(drift_df):
    # First compute per-firm mean JSD across all firms.
    firm_stats = drift_df.groupby("firm_name").agg(
        avg_lda_jsd=("lda_jsd", "mean"),
        avg_sbert_drift=("sbert_drift", "mean"),
        n_year_pairs=("lda_jsd", "size"),
    ).reset_index()

    # Derive tercile cutoffs of avg_lda_jsd: bottom 33% = Minimal, middle = Moderate, top = Significant.
    t33, t67 = firm_stats["avg_lda_jsd"].quantile([1/3, 2/3]).values
    log.info(f"JSD tercile cutoffs (data-driven): t33 = {t33:.4f}, t67 = {t67:.4f}")

    def label_for(jsd):
        if jsd >= t67:  return "Significant topic shifts"
        if jsd >= t33:  return "Moderate topic shifts"
        return "Minimal topic shifts"

    firm_stats["topic_shift_label"] = firm_stats["avg_lda_jsd"].apply(label_for)
    firm_stats["tercile_cutoff_t33"] = round(float(t33), 4)
    firm_stats["tercile_cutoff_t67"] = round(float(t67), 4)

    for col in ["avg_lda_jsd", "avg_sbert_drift"]:
        firm_stats[col] = firm_stats[col].round(4)

    return firm_stats

qual = build_qualitative_summary(drift)
qual.to_csv(cfg.VERIS_QUALITATIVE_CSV, index=False)
log.info(f"Qualitative summary saved: {cfg.VERIS_QUALITATIVE_CSV.name} ({len(qual)} rows)")
display(qual)

19:22:05 [INFO] VERIS -- JSD tercile cutoffs (data-driven): t33 = 0.1134, t67 = 0.2091
19:22:05 [INFO] VERIS -- Qualitative summary saved: veris_qualitative_summary.csv (12 rows)


,firm_name,avg_lda_jsd,avg_sbert_drift,n_year_pairs,topic_shift_label,tercile_cutoff_t33,tercile_cutoff_t67
0,BP,0.1368,0.0113,10,Moderate topic shifts,0.1134,0.2091
1,Chevron,0.1976,0.0243,10,Moderate topic shifts,0.1134,0.2091
2,ConocoPhillips,0.2322,0.0377,10,Significant topic shifts,0.1134,0.2091
3,Eni,0.1108,0.0215,9,Minimal topic shifts,0.1134,0.2091
4,Equinor,0.1134,0.0193,10,Minimal topic shifts,0.1134,0.2091
5,ExxonMobil,0.4806,0.0668,3,Significant topic shifts,0.1134,0.2091
6,Glencore,0.0894,0.0133,10,Minimal topic shifts,0.1134,0.2091
7,Repsol,0.1134,0.0103,7,Moderate topic shifts,0.1134,0.2091
8,RioTinto,0.3661,0.0592,10,Significant topic shifts,0.1134,0.2091
9,Shell,0.0901,0.0210,9,Minimal topic shifts,0.1134,0.2091
